# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
from pathlib import Path
from collections import Counter
import re
import tarfile
import zipfile

import nltk
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

pd.set_option("display.max_colwidth", 160)

In [2]:
ruta_zip = Path("C:/Users/VICTUS/Desktop/EPN/RI/txt-files.tar.zip")
max_documentos = 1000

try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    nltk.download("punkt_tab", quiet=True, raise_on_error=True)

if not ruta_zip.is_file():
    raise FileNotFoundError(ruta_zip)

In [3]:
def preprocesar(texto):
    tokens = nltk.word_tokenize(texto)
    return [t.lower() for t in tokens if t.isalpha()]


def obtener_titulo(texto, doc_id):
    coincidencia = re.search(
        r"^Title:\s*(.+)", texto, flags=re.MULTILINE | re.IGNORECASE
    )
    return coincidencia.group(1).strip() if coincidencia else f"Gutenberg {doc_id}"

In [4]:
corpus_original = {}
titulos = {}
patron_id = re.compile(r"(\d+)")

with zipfile.ZipFile(ruta_zip) as archivo_zip:
    nombre_tar = next(n for n in archivo_zip.namelist() if n.endswith(".tar"))
    with archivo_zip.open(nombre_tar) as contenido_tar:
        with tarfile.open(fileobj=contenido_tar, mode="r|") as archivo_tar:
            for miembro in archivo_tar:
                if not miembro.isfile() or not miembro.name.endswith(".txt"):
                    continue
                coincidencia = patron_id.search(miembro.name)
                if coincidencia is None:
                    continue
                doc_id = coincidencia.group(1)
                if doc_id in corpus_original:
                    continue
                with archivo_tar.extractfile(miembro) as archivo:
                    texto = archivo.read().decode("utf-8", errors="ignore")
                corpus_original[doc_id] = texto
                titulos[doc_id] = obtener_titulo(texto, doc_id)
                if len(corpus_original) == max_documentos:
                    break

assert len(corpus_original) == max_documentos
print("Documentos cargados:", len(corpus_original))

Documentos cargados: 1000


In [5]:
corpus_procesado = {}
frecuencias = {}

for doc_id, texto in corpus_original.items():
    tokens = preprocesar(texto)
    corpus_procesado[doc_id] = " ".join(tokens)
    frecuencias[doc_id] = Counter(tokens)

del tokens
ids_documentos = list(corpus_original)
longitudes = np.array([sum(frecuencias[d].values()) for d in ids_documentos])
N = len(ids_documentos)
vocabulario = sorted(set().union(*(f.keys() for f in frecuencias.values())))
indice_terminos = {termino: i for i, termino in enumerate(vocabulario)}

In [6]:
tabla_documentos = pd.DataFrame({
    "Identificador": ids_documentos,
    "Título": [titulos[d] for d in ids_documentos],
    "Número de términos únicos": [len(frecuencias[d]) for d in ids_documentos]
})
display(tabla_documentos)
display(pd.DataFrame([{
    "Documentos": N,
    "Vocabulario": len(vocabulario),
    "Longitud mínima": int(longitudes.min()),
    "Longitud media": longitudes.mean(),
    "Longitud máxima": int(longitudes.max())
}]))

,Identificador,Título,Número de términos únicos
0,10000,The Magna Carta,1770
1,10001,Apocolocyntosis,2020
2,10002,The House on the Borderland,5199
3,10003,"My First Years as a Frenchwoman, 1876-1879",6061
4,10004,The Warriors,6565
...,...,...,...
995,11079,Essays in Rebellion,11374
996,1107,The Taming of the Shrew,4009
997,11080,"The Orations of Marcus Tullius Cicero, Volume 4",9203
998,11081,Extracto de la gramatica mutsun,2703


,Documentos,Vocabulario,Longitud mínima,Longitud media,Longitud máxima
0,1000,611592,3118,70157.368,976132


Se utilizan los primeros 1.000 identificadores distintos del archivo `txt-files.tar.zip`, con el mismo criterio de selección de la parte B de la práctica 2. Se conserva la primera versión TXT encontrada de cada libro.

Se aplica `nltk.word_tokenize`, se conservan los tokens alfabéticos y se convierten a minúsculas. No se eliminan stopwords ni se aplica stemming. Las longitudes corresponden a los tokens resultantes. Se conserva el texto original para los fragmentos.

### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [7]:
def calcular_tf(termino, frecuencias_documento):
    return frecuencias_documento.get(termino, 0)


def calcular_df(frecuencias_corpus):
    df = Counter()
    for frecuencias_documento in frecuencias_corpus.values():
        df.update(frecuencias_documento.keys())
    return df


def calcular_idf(df, numero_documentos):
    return {t: np.log(numero_documentos / frecuencia) + 1 for t, frecuencia in df.items()}


def calcular_tfidf(termino, frecuencias_documento, idf):
    return calcular_tf(termino, frecuencias_documento) * idf.get(termino, 0.0)

In [8]:
df_corpus = calcular_df(frecuencias)
idf_corpus = calcular_idf(df_corpus, N)

seis_pesos = []
for doc_id in ids_documentos[:2]:
    for termino, _ in frecuencias[doc_id].most_common(3):
        seis_pesos.append({
            "Identificador": doc_id,
            "Término": termino,
            "TF": calcular_tf(termino, frecuencias[doc_id]),
            "DF": df_corpus[termino],
            "IDF": idf_corpus[termino],
            "TF-IDF": calcular_tfidf(termino, frecuencias[doc_id], idf_corpus)
        })
display(pd.DataFrame(seis_pesos))

,Identificador,Término,TF,DF,IDF,TF-IDF
0,10000,the,1144,1000,1.0,1144.0
1,10000,of,900,1000,1.0,900.0
2,10000,and,662,1000,1.0,662.0
3,10001,the,460,1000,1.0,460.0
4,10001,of,248,1000,1.0,248.0
5,10001,to,233,1000,1.0,233.0


Los seis pesos corresponden a tres términos de cada uno de los dos primeros documentos. TF cuenta apariciones dentro del documento; DF cuenta documentos que contienen el término. Un término presente en todos los documentos tiene IDF igual a 1. Cuanto menor es su DF, mayor es su IDF.

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [9]:
filas = []
columnas = []
valores = []

for fila, doc_id in enumerate(ids_documentos):
    for termino, frecuencia in frecuencias[doc_id].items():
        filas.append(fila)
        columnas.append(indice_terminos[termino])
        valores.append(frecuencia)

matriz_tf = csr_matrix(
    (valores, (filas, columnas)),
    shape=(N, len(vocabulario)),
    dtype=np.float64
)
del filas, columnas, valores

df_vector = np.asarray((matriz_tf > 0).sum(axis=0)).ravel()
idf_vector = np.array([idf_corpus[t] for t in vocabulario])
np.testing.assert_array_equal(df_vector, [df_corpus[t] for t in vocabulario])

In [10]:
matriz_tfidf = matriz_tf.multiply(idf_vector).tocsr()

print("Dimensiones documento-término:", matriz_tfidf.shape)
print("Pesos distintos de cero:", matriz_tfidf.nnz)
print("Densidad:", matriz_tfidf.nnz / np.prod(matriz_tfidf.shape))
display(pd.DataFrame(
    matriz_tfidf[:5, :8].toarray(),
    index=ids_documentos[:5],
    columns=vocabulario[:8]
))

Dimensiones documento-término: (1000, 611592)
Pesos distintos de cero: 6597387
Densidad: 0.010787235608052427


,a,aa,aaa,aaaaaah,aaaarranger,aaaffaires,aaalors,aaannoncer
10000,206.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10001,184.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10002,1425.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10003,1498.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10004,1362.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [11]:
vectorizador = TfidfVectorizer(
    analyzer=str.split,
    vocabulary=indice_terminos,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    dtype=np.float64
)
matriz_tfidf_sklearn = vectorizador.fit_transform(
    corpus_procesado[d] for d in ids_documentos
)

C:\Users\VICTUS\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:1368: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


In [12]:
diferencia = (matriz_tfidf - matriz_tfidf_sklearn).tocsr()
max_diferencia = float(np.max(np.abs(diferencia.data))) if diferencia.nnz else 0.0
tolerancia = 1e-10

np.testing.assert_allclose(idf_vector, vectorizador.idf_, rtol=0, atol=tolerancia)
assert max_diferencia <= tolerancia
print("Máxima diferencia absoluta:", max_diferencia)
print("Tolerancia absoluta:", tolerancia)

Máxima diferencia absoluta: 0.0
Tolerancia absoluta: 1e-10


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [13]:
consultas = pd.DataFrame([
    ("Q1", "sea ship captain whale", "Relatos de navegación, capitanes y caza de ballenas."),
    ("Q2", "science evolution species nature", "Textos sobre ciencia, evolución y diversidad de especies."),
    ("Q3", "love marriage family", "Historias sobre relaciones de pareja y conflictos familiares."),
    ("Q4", "war army battle soldier", "Relatos militares sobre batallas y vida de los soldados.")
], columns=["Consulta", "Texto", "Necesidad de información"])
consultas["Tokens"] = consultas["Texto"].apply(preprocesar)
display(consultas)

,Consulta,Texto,Necesidad de información,Tokens
0,Q1,sea ship captain whale,"Relatos de navegación, capitanes y caza de ballenas.","[sea, ship, captain, whale]"
1,Q2,science evolution species nature,"Textos sobre ciencia, evolución y diversidad de especies.","[science, evolution, species, nature]"
2,Q3,love marriage family,Historias sobre relaciones de pareja y conflictos familiares.,"[love, marriage, family]"
3,Q4,war army battle soldier,Relatos militares sobre batallas y vida de los soldados.,"[war, army, battle, soldier]"


In [14]:
def vectorizar_consulta(tokens):
    conteos = Counter(tokens)
    posiciones = [indice_terminos[t] for t in conteos if t in indice_terminos]
    pesos = [conteos[t] * idf_corpus[t] for t in conteos if t in indice_terminos]
    return csr_matrix(
        (pesos, ([0] * len(posiciones), posiciones)),
        shape=(1, len(vocabulario))
    )


vectores_consulta = [vectorizar_consulta(tokens) for tokens in consultas["Tokens"]]
vectores_consulta_sklearn = vectorizador.transform(
    " ".join(tokens) for tokens in consultas["Tokens"]
)
for i, vector in enumerate(vectores_consulta):
    np.testing.assert_allclose(
        vector.toarray(), vectores_consulta_sklearn[i].toarray(),
        rtol=0, atol=tolerancia
    )

consultas["Fuera del vocabulario"] = consultas["Tokens"].apply(
    lambda tokens: [t for t in tokens if t not in indice_terminos]
)
display(consultas[["Consulta", "Fuera del vocabulario"]])

,Consulta,Fuera del vocabulario
0,Q1,[]
1,Q2,[]
2,Q3,[]
3,Q4,[]


In [15]:
def similitud_coseno_producto(matriz_documentos, vector_consulta):
    productos = (matriz_documentos @ vector_consulta.T).toarray().ravel()
    normas_documentos = np.sqrt(np.asarray(
        matriz_documentos.multiply(matriz_documentos).sum(axis=1)
    ).ravel())
    norma_consulta = np.sqrt(vector_consulta.multiply(vector_consulta).sum())
    denominadores = normas_documentos * norma_consulta
    return np.divide(
        productos, denominadores,
        out=np.zeros_like(productos), where=denominadores > 0
    )

In [16]:
puntuaciones_tfidf = np.column_stack([
    similitud_coseno_producto(matriz_tfidf, vector)
    for vector in vectores_consulta
])
puntuaciones_sklearn = cosine_similarity(
    matriz_tfidf_sklearn, vectores_consulta_sklearn
)
np.testing.assert_allclose(
    puntuaciones_tfidf, puntuaciones_sklearn, rtol=0, atol=tolerancia
)
print("Máxima diferencia en cosenos:", np.abs(
    puntuaciones_tfidf - puntuaciones_sklearn
).max())

Máxima diferencia en cosenos: 1.1379786002407855e-15


In [17]:
i_consulta = 0
i_documento = int(np.argmax(puntuaciones_tfidf[:, i_consulta]))
vector_q = vectores_consulta[i_consulta]
vector_d = matriz_tfidf[i_documento]
compartidos = sorted(set(vector_q.indices) & set(vector_d.indices))

display(pd.DataFrame([{
    "Término compartido": vocabulario[j],
    "Peso consulta": vector_q[0, j],
    "Peso documento": vector_d[0, j],
    "Producto parcial": vector_q[0, j] * vector_d[0, j]
} for j in compartidos]))

producto = float((vector_q @ vector_d.T)[0, 0])
norma_q = float(np.sqrt(vector_q.multiply(vector_q).sum()))
norma_d = float(np.sqrt(vector_d.multiply(vector_d).sum()))
display(pd.DataFrame([{
    "Consulta": consultas.iloc[i_consulta]["Consulta"],
    "Identificador": ids_documentos[i_documento],
    "Producto punto": producto,
    "Norma consulta": norma_q,
    "Norma documento": norma_d,
    "Similitud": producto / (norma_q * norma_d)
}]))

,Término compartido,Peso consulta,Peso documento,Producto parcial
0,sea,1.236989,85.352238,105.579776
1,ship,1.510826,3.021651,4.565188
2,whale,3.024953,105.873367,320.261998


,Consulta,Identificador,Producto punto,Norma consulta,Norma documento,Similitud
0,Q1,10617,430.406963,3.947294,1913.550831,0.056982


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [18]:
def obtener_fragmento(doc_id, tokens_consulta, longitud=280):
    texto = corpus_original[doc_id]
    inicio_cuerpo = re.search(
        r"\*\*\* START OF[^\n]*", texto, flags=re.IGNORECASE
    )
    cuerpo = texto[inicio_cuerpo.end():] if inicio_cuerpo else texto
    patron = r"\b(?:" + "|".join(re.escape(t) for t in tokens_consulta) + r")\b"
    coincidencia = re.search(patron, cuerpo, flags=re.IGNORECASE) if tokens_consulta else None
    inicio = max(0, coincidencia.start() - 70) if coincidencia else 0
    fragmento = re.sub(r"\s+", " ", cuerpo[inicio:inicio + longitud]).strip()
    return ("…" if inicio else "") + fragmento + ("…" if inicio + longitud < len(cuerpo) else "")


def construir_ranking(puntuaciones, tokens_consulta, k=5):
    orden = np.argsort(-puntuaciones, kind="stable")[:k]
    return pd.DataFrame([{
        "Posición": posicion,
        "Identificador": ids_documentos[i],
        "Título": titulos[ids_documentos[i]],
        "Puntuación": puntuaciones[i],
        "Fragmento": obtener_fragmento(ids_documentos[i], tokens_consulta)
    } for posicion, i in enumerate(orden, start=1)])

In [19]:
rankings_tfidf = {}

for j, consulta in consultas.iterrows():
    ranking = construir_ranking(puntuaciones_tfidf[:, j], consulta["Tokens"])
    ranking_sklearn = construir_ranking(puntuaciones_sklearn[:, j], consulta["Tokens"])
    np.testing.assert_allclose(
        ranking["Puntuación"], ranking_sklearn["Puntuación"], rtol=0, atol=tolerancia
    )
    assert ranking["Identificador"].tolist() == ranking_sklearn["Identificador"].tolist()
    rankings_tfidf[consulta["Consulta"]] = ranking
    print(consulta["Consulta"], "—", consulta["Texto"])
    display(ranking)

Q1 — sea ship captain whale


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,10617,Within the Deep,0.056982,…SOME STRANGE NURSERIES V. THE OGRE OF THE DEEP VI. THE WHALE VII. TIGERS OF THE SEA VIII. THE DANGERS OF THE DEEP IX. THE FISH OF OUR ROCK-POOLS X. SOME CU...
1,2,10871,"At Sunwich Port, Part 1.",0.047632,"…h, and Sunwich is proud of it. The tall grey tower is a landmark at sea, but from the narrow streets of the little town itself it has a disquieting appeara..."
2,3,10339,An Antarctic Mystery,0.044893,…ds. Chapter II. The Schooner _Halbrane_ Chapter III. Captain Len Guy Chapter IV. From the Kerguelen Isles to Prince Edward Island Chapter V. Edgar Poe's Ro...
3,4,1058,The Mirror of the Sea,0.040450,…x074@pglaf.org THE MIRROR OF THE SEA MEMORIES AND IMPRESSIONS BY JOSEPH CONRAD * * * * * “ . . . for th…
4,5,10008,The Mystery,0.036873,"…ions by Will Crawford_ 1907 CONTENTS PART ONE THE SEA RIDDLE I. DESERT SEAS II. THE ""LAUGHING LASS"" III. THE DEATH SHIP IV. THE SECOND PRIZE CREW V. THE DI..."


Q2 — science evolution species nature


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,10852,Hardy Ornamental Flowering Trees and Shrubs,0.065761,"…r desirable. There is quite a host of choice and beautiful flowering species, which, though at present not generally known are yet perfectly hardy, of the ..."
1,2,10437,Pulpit and Press (6th Edition),0.035584,"…Y REVEREND MARY BAKER EDDY, DISCOVERER AND FOUNDER OF CHRISTIAN SCIENCE. 1897. CONTENTS DEDICATORY SERMON CHRISTIAN SCIENCE TEXT-BOOK HYMN--_Laying the Cor..."
2,3,10427,Scientific Essays and Lectures,0.030973,…On Bio-Geology The Study of Natural History Superstition Science Thoughts in a Gravel-Pit How to Study Natural History The Natural Theology of the Future O...
3,4,10616,"An Essay Concerning Humane Understanding, Volume 2",0.027667,"…e great instrument and common tie of society. Man, therefore, had by nature his organs so fashioned, as to be fit to frame articulate sounds, which we call..."
4,5,10139,The Faith of the Millions (2nd series),0.025180,"…ascent to God but by the ladder of creatures; he goes to the book of Nature and of human life, and to the book of Revelation, and turns and ponders their p..."


Q3 — love marriage family


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,1105,The Sonnets,0.064203,"…y husbandry? Or who is he so fond will be the tomb, Of his self-love to stop posterity? Thou art thy mother's glass and she in thee Calls back the lovely A..."
1,2,1041,Shakespeare's Sonnets,0.062387,"…f thy husbandry? Or who is he so fond will be the tomb, Of his self-love to stop posterity? Thou art thy mother’s glass and she in thee Calls back the love..."
2,3,10063,Happiness and Marriage,0.046531,"HAPPINESS AND MARRIAGE BY ELIZABETH TOWNE ""The inner side of every cloud Is bright and shining; I therefore turn my clouds about, And always wear them insid..."
3,4,10913,English Poems,0.045026,"…er: Hear the conclusion of the whole matter. You dream like mad, you love like tinder, you aspire like a star-struck moth--for what? That you may hive litt..."
4,5,1045,Venus and Adonis,0.040304,"…n, Rose-cheek’d Adonis hied him to the chase; Hunting he lov’d, but love he laugh’d to scorn; 4 Sick-thoughted Venus makes amain unto him, And like a bold-..."


Q4 — war army battle soldier


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,10629,Britain at Bay,0.052709,…RIGHT IV. ARBITRATION AND DISARMAMENT V. THE NATIONALISATION OF WAR VI. THE BALANCE OF POWER VII. THE RISE OF GERMANY VIII. NATIONHOOD NEGLECTED IX. NEW CO...
1,2,10692,A Life of Gen. Robert E. Lee,0.051498,…-Stratford V.--Lee's Early Manhood and Career in the United States Army VI.--Lee and Scott VII.--Lee resigns VIII.--His Reception at Richmond IX.--Lee in 1...
2,3,10147,America's War for Humanity,0.046323,"AMERICA'S WAR FOR HUMANITY Pictorial History _of the_ World War _for_ Liberty _By_ THOMAS H. RUSSELL, A.M. LL.D. _Noted Historical and Military Writer. Memb..."
3,4,1068,Personal Memoirs of U. S. Grant — Volume 2,0.044246,…LI. ASSUMING THE COMMAND AT CHATTANOOGA--OPENING A LINE OF SUPPLIES--BATTLE OF WAUHATCHIE--ON THE PICKET LINE. CHAPTER XLII. CONDITION OF THE ARMY--REBUILD...
4,5,10159,England and the War,0.039885,ENGLAND AND THE WAR being SUNDRY ADDRESSES delivered during the war and now first collected by WALTER RALEIGH OXFORD 1918 CONTENTS PREFACE MIGHT IS RIGHT Fi...
